# Export to Ranged Summarized Experiment

#### Load packages

In [ ]:
#| eval: true
#| message: false 
#| warning: false

library(reticulate)
use_python("/home/adaml9/scratch/miniforge3/envs/scFates/bin/python", required = TRUE)

library(anndata)
library(Seurat)
library(SummarizedExperiment)
library(GenomicRanges)
library(configr)

set.seed(1)

# Memory and compatibility options
options(
  future.globals.maxSize = 3e+09,
  Seurat.object.assay.version = "v3"
)

In [ ]:
config <- read.config("/pstore/data/ihb-g-deco/USERS/adaml9/intestine_fate/notebooks/global_analysis/multiome/grn/config.ini")

In [ ]:
data_dir <- config$DEFAULT$PROCESSED_DATA_DIR

#### Load data

In [ ]:
# Load AnnData object
adata <- read_h5ad(file.path(data_dir, "10X.multiome.processed.controls.integrated.h5ad"))

#### Convert

In [ ]:
library(tidyverse)
library(org.Hs.eg.db)
library(AnnotationDbi)

library(anndata)
library(Matrix)
library(GenomicRanges)
library(SummarizedExperiment)
library(org.Hs.eg.db)
library(AnnotationDbi)
library(dplyr)

# Convert to sparse matrix and transpose (cells x genes → genes x cells)
counts <- as(adata$X, "dgCMatrix")
counts <- t(counts)

# Assign names
rownames(counts) <- adata$var_names
colnames(counts) <- adata$obs_names

# Map gene symbols to genomic coordinates 
gene_symbols <- rownames(counts)

mapped <- AnnotationDbi::select(
  org.Hs.eg.db,
  keys = gene_symbols,
  columns = c("CHR", "CHRLOC", "CHRLOCEND"),
  keytype = "SYMBOL"
)

In [ ]:
# Filter valid rows
mapped_clean <- mapped %>%
  dplyr::filter(!is.na(CHRLOC) & !is.na(CHRLOCEND)) %>%
  dplyr::filter(CHR == CHRLOCCHR) %>%
  mutate(
    strand = ifelse(CHRLOC < 0 | CHRLOCEND < 0, "-", "+"),
    start = pmin(abs(CHRLOC), abs(CHRLOCEND)),
    end = pmax(abs(CHRLOC), abs(CHRLOCEND)),
    chr = paste0("chr", CHR)
  ) %>%
  group_by(SYMBOL) %>%
  slice(1) %>%
  ungroup()

# Filter count matrix to mapped genes
genes_mapped <- mapped_clean$SYMBOL
counts_mapped <- counts[rownames(counts) %in% genes_mapped, , drop = FALSE]

# Keep same order in metadata
mapped_clean <- mapped_clean[match(rownames(counts_mapped), mapped_clean$SYMBOL), ]

# Build GRanges object
row_ranges <- GRanges(
  seqnames = mapped_clean$chr,
  ranges = IRanges(start = mapped_clean$start, end = mapped_clean$end),
  strand = mapped_clean$strand,
  gene_name = mapped_clean$SYMBOL
)
names(row_ranges) <- mapped_clean$SYMBOL

# Build colData 
col_data <- adata$obs
rownames(col_data) <- adata$obs_names

col_data$Clusters <- as.character(col_data$cell_type_merged)

# Create RangedSummarizedExperiment
rse <- SummarizedExperiment(
  assays = list(counts = counts_mapped),
  rowRanges = row_ranges,
  colData = col_data
)

In [ ]:
saveRDS(rse, "/pstore/data/ihb-g-deco/USERS/adaml9/nf-core-scatacpipe/data/scRNA-EEC.rds")